# Lab 21.2: Auditing HermAI 0.1.8 on One Refused Applicant

Chapter 21, The Hermeneutic Circle. Run the cells in order; checkpoints tell you what you should see.

*Disclosure: the author of the book develops and maintains HermAI. This lab audits the released version (0.1.8, October 2025) exactly as published, with the same checks you would apply to any other explainer.*

Data: German Credit (Statlog), from OpenML (`credit-g`); see `data/german-credit.md`. The helper module `../lab3/hct_mini.py` loads the data, trains the random forest and defines the horizon and the claim test, so this lab and Lab 21.3 use identical code.

In [1]:
import sys, io, contextlib, inspect
import numpy as np, pandas as pd
sys.path.insert(0, "../lab3")
import hct_mini as h
import hermai.perturbations as hp_pkg
import hermai.perturbations.tabular as hp_tab
from hermai.explainers import LocalExplainer, GeneralExplainer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text

S = h.setup(0)
f, Xte, cols, cb = S["f"], S["Xte"].values, S["cols"], S["codebook"]
print("test AUC:", round(roc_auc_score(S["yte"], f(Xte)), 3))
H = h.Horizon(S["Xfit"], S["Xcal"])
print("real test records inside the horizon:", round(H.inside(Xte).mean(), 2))

test AUC: 0.787
real test records inside the horizon: 0.92


**Checkpoint 1:** test AUC 0.787; 0.92 of real test records lie inside the horizon (the rest fall outside the training range of some feature).

## 1. Which generator do you get?

HermAI 0.1.8 ships two classes with the same name. The import line decides which one you use.

In [2]:
print("package level:", hp_pkg.TabularPerturbationGenerator.__module__)
print("README path:  ", hp_tab.TabularPerturbationGenerator.__module__)
print("same class?   ", hp_pkg.TabularPerturbationGenerator is hp_tab.TabularPerturbationGenerator)
print("seed argument in perturb():", "seed" in inspect.signature(hp_pkg.TabularPerturbationGenerator.perturb).parameters)

package level: hermai.perturbations.tabular_generator
README path:   hermai.perturbations.tabular
same class?    False
seed argument in perturb(): False


**Checkpoint 2:** two different modules (`tabular_generator` and `tabular`), not the same class, and no seed argument. The package-level class adds independent noise of one quarter of each feature's standard deviation; the other draws correlated noise from one quarter of the covariance. Both resample every categorical feature from its overall frequencies, whatever the applicant's own value.

## 2. Explain the featured applicant

In [3]:
i = 77                                 # test row of the applicant in the chapter's case study
x = Xte[i]
print("predicted risk of bad credit:", round(f(x[None])[0], 3))
print({c: (cb[c][int(v)] if c in cb else v) for c, v in zip(cols, x)})

def explain(gen_cls, seed):
    g = gen_cls(categorical_features=S["cat"])
    with contextlib.redirect_stdout(io.StringIO()):
        g.fit(S["Xfit"])
    rec = h.Recorder(f)
    np.random.seed(seed)
    with contextlib.redirect_stdout(io.StringIO()):
        ex = LocalExplainer(rec, g).explain(pd.Series(x, index=cols), n_samples=1000)
    return ex, rec.evidence(skip_first=True)

ex, E = explain(hp_pkg.TabularPerturbationGenerator, seed=77)   # the seed Lab 21.3 uses for this applicant
print(ex.narrative())

predicted risk of bad credit: 0.668
{'checking_status': '<0', 'duration': np.float64(15.0), 'credit_history': 'no credits/all paid', 'purpose': 'new car', 'credit_amount': np.float64(950.0), 'savings_status': '<100', 'employment': '>=7', 'installment_commitment': np.float64(4.0), 'personal_status': 'male single', 'other_parties': 'none', 'residence_since': np.float64(3.0), 'property_magnitude': 'car', 'age': np.float64(33.0), 'other_payment_plans': 'none', 'housing': 'rent', 'existing_credits': np.float64(2.0), 'job': 'skilled', 'num_dependents': np.float64(2.0), 'own_telephone': 'none', 'foreign_worker': 'yes'}


Prediction Analysis for Instance:
---------------------------------
Model predicted class '1' with 66.83% confidence.

Key Feature Contributions (Local Importance):
- Feature 'checking_status' decreases the probability.
- Feature 'duration' increases the probability.
- Feature 'purpose' decreases the probability.
- Feature 'own_telephone' decreases the probability.
- Feature 'credit_amount' decreases the probability.

Counterfactual Analysis:
To flip the prediction to class '0', one of the closest possibilities found is:
- Change 'checking_status' from 1.00 to 0.00
- Change 'duration' from 15.00 to 13.16
- Change 'credit_history' from 4.00 to 3.00
- Change 'purpose' from 4.00 to 2.00
- Change 'credit_amount' from 950.00 to 800.80
- Change 'savings_status' from 2.00 to 3.00
- Change 'installment_commitment' from 4.00 to 3.98
- Change 'residence_since' from 3.00 to 3.05
- Change 'age' from 33.00 to 30.04
- Change 'housing' from 2.00 to 1.00
- Change 'existing_credits' from 2.00 to 2.13
-

**Checkpoint 3:** predicted risk 0.668. The narrative reports class 1 "with 66.83% confidence", says that `checking_status` *decreases* the probability and `duration` *increases* it, and proposes a counterfactual that changes about a dozen features, several to fractional values (for example 13.16 months, 2.13 existing credits).

Note: the percentage the narrative calls "confidence" is the probability of class 1, whatever class is predicted; read the source of `LocalExplanation.narrative` to see it.

## 3. Is the evidence made of possible cases?

In [4]:
d_ok, t_ok = H.parts(E)
print("evidence points:", len(E))
print("inside by distance only:", round(d_ok.mean(), 3))
print("inside by measurement scale only (integers, observed range):", round(t_ok.mean(), 3))
print("Horizon Plausibility (both):", round((d_ok & t_ok).mean(), 3))
print("example evidence row:", np.round(E[0], 2))

evidence points: 1000
inside by distance only: 0.946
inside by measurement scale only (integers, observed range): 0.0
Horizon Plausibility (both): 0.0
example evidence row: [1.00000e+00 1.56600e+01 1.00000e+00 0.00000e+00 1.40064e+03 2.00000e+00
 0.00000e+00 3.83000e+00 3.00000e+00 2.00000e+00 3.11000e+00 3.00000e+00
 3.43800e+01 1.00000e+00 2.00000e+00 1.71000e+00 1.00000e+00 2.03000e+00
 0.00000e+00 1.00000e+00]


**Checkpoint 4:** most evidence lies close to real records by distance (0.946), but none respects the measurement scale: every point carries fractional months, installment rates or numbers of dependents, so HP is 0.

## 4. Two readings of one sentence

In [5]:
coef = ex.feature_importances.set_index("feature").loc[cols, "importance"].values
mu = S["Xfit"].values.mean(0)
slope_raw = coef / E.std(0)
contrib = slope_raw * (x - mu)
rows = []
for j in h.top3(coef):
    ok_nat, delta = h.claim_test(S, H, x, j, np.sign(coef[j]))
    rows.append(dict(feature=cols[j], narrative_says="increases" if coef[j] > 0 else "decreases",
                     contribution_sign="+" if contrib[j] > 0 else "-",
                     on_horizon_delta=round(delta, 3), natural_reading_passes=ok_nat,
                     slope_reading_passes=h.slope_test(S, H, x, j, np.sign(coef[j]))))
pd.DataFrame(rows)

,feature,narrative_says,contribution_sign,on_horizon_delta,natural_reading_passes,slope_reading_passes
0,checking_status,decreases,+,0.130,False,True
1,duration,increases,-,-0.015,False,True
2,purpose,decreases,+,0.051,False,True


**Checkpoint 5:** all three sentences pass the literal slope reading, and none passes the on-horizon test under the natural reading (the value of this feature pushed this prediction up or down). For `checking_status` the measured effect is +0.130: replacing "below 0 DM" with the values of comparable applicants lowers the risk, so the account status raised it. The narrative says the opposite. Note also that `checking_status` and `purpose` are integer codes for categories: a slope on a code number has no meaning, which is one more reason to distrust the sentence.

## 5. Run to run

In [6]:
tops, signs = [], []
for s in range(5):
    e, _ = explain(hp_pkg.TabularPerturbationGenerator, seed=1000 + s)
    c = e.feature_importances.set_index("feature").loc[cols, "importance"].values
    t = h.top3(c); tops.append(set(t)); signs.append({j: np.sign(c[j]) for j in t})
jac = [len(a & b) / len(a | b) for k, a in enumerate(tops) for b in tops[k + 1:]]
agree = [np.mean([signs[k][j] == signs[m][j] for j in tops[k] & tops[m]]) for k in range(5) for m in range(k + 1, 5) if tops[k] & tops[m]]
print("top-3 sets:", [sorted(cols[j] for j in t) for t in tops])
print("mean Jaccard of top-3 sets:", round(np.mean(jac), 2), "| sign agreement on shared features:", round(np.mean(agree), 2))

top-3 sets: [['checking_status', 'own_telephone', 'purpose'], ['checking_status', 'duration', 'purpose'], ['checking_status', 'credit_amount', 'purpose'], ['checking_status', 'duration', 'purpose'], ['checking_status', 'duration', 'purpose']]
mean Jaccard of top-3 sets: 0.65 | sign agreement on shared features: 1.0


**Checkpoint 6:** mean Jaccard 0.65 and sign agreement 1.0: across five random states the narrative always names `checking_status` and `purpose` and always gives them the same (wrong) direction, while the third feature changes from run to run. Stability is not correctness.

## 6. The whole: the general explainer

In [7]:
class Clf:                                   # GeneralExplainer calls predict()
    def predict(self, A): return (f(np.asarray(A, float)) >= 0.5).astype(int)
gen = GeneralExplainer(Clf()).explain(S["Xfit"], max_depth=3)
tree = gen.surrogate_model
fid = (tree.predict(S["Xte"]) == Clf().predict(Xte)).mean()
print("surrogate agrees with the forest's decisions on the test set:", round(fid, 3))
print(export_text(tree, feature_names=cols))

surrogate agrees with the forest's decisions on the test set: 0.807
|--- checking_status <= 1.50
|   |--- purpose <= 4.50
|   |   |--- employment <= 1.50
|   |   |   |--- class: 0
|   |   |--- employment >  1.50
|   |   |   |--- class: 1
|   |--- purpose >  4.50
|   |   |--- credit_amount <= 9790.00
|   |   |   |--- class: 0
|   |   |--- credit_amount >  9790.00
|   |   |   |--- class: 1
|--- checking_status >  1.50
|   |--- credit_amount <= 7839.50
|   |   |--- other_payment_plans <= 0.50
|   |   |   |--- class: 0
|   |   |--- other_payment_plans >  0.50
|   |   |   |--- class: 0
|   |--- credit_amount >  7839.50
|   |   |--- purpose <= 3.50
|   |   |   |--- class: 1
|   |   |--- purpose >  3.50
|   |   |   |--- class: 0



**Checkpoint 7:** the depth-3 tree agrees with the forest on 0.807 of test decisions; read its rules and find the leaf of the featured applicant. The tree splits on integer codes of categories, so a rule such as `checking_status <= 1.5` must be translated back with the codebook (`S["codebook"]`) before anyone reads it.

## Your turn

Rewrite the narrative of Section 2 in three sentences that (a) state contributions, not slopes, (b) use category names, not codes, and (c) pass the on-horizon test of Section 4. Then write one sentence on what a counterfactual for this applicant should look like (few changes, possible values only).